# Bab 8: Naïve Bayes

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab08/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab08")
print("Folder kerja:", os.getcwd())

## Listing 8.1: Naive Bayes ditulis sendiri

`kode/bab08/bayes_mini.py`

In [ ]:
import pandas as pd

mini = pd.DataFrame({
    "tingkat":   ["perunggu", "perunggu", "perak", "perunggu", "perak",
                  "perunggu", "perak", "perak", "perunggu", "perak"],
    "kunjungan": ["jarang", "jarang", "jarang", "sering", "sering",
                  "sering", "sering", "jarang", "sering", "sering"],
    "keluhan":   ["ya", "tidak", "ya", "ya", "tidak",
                  "tidak", "tidak", "tidak", "ya", "tidak"],
    "promo":     ["tinggi", "tinggi", "tinggi", "tinggi", "rendah",
                  "tinggi", "rendah", "rendah", "rendah", "tinggi"],
    "churn":     ["ya", "ya", "ya", "ya", "tidak",
                  "tidak", "tidak", "tidak", "tidak", "tidak"]})


def naive_bayes(data, x, alfa=0):
    prior = data["churn"].value_counts(normalize=True)
    skor = {}
    for c in prior.index:
        bagian = data[data["churn"] == c]
        p = prior[c]
        for atr, nilai in x.items():
            k = data[atr].nunique()
            p *= ((bagian[atr] == nilai).sum() + alfa) / (len(bagian)
                                                          + alfa * k)
        skor[c] = p
    total = sum(skor.values())
    return " ".join(f"{c}: {s:.6f} -> {s / total:.4f}"
                    for c, s in skor.items())


x1 = {"tingkat": "perunggu", "kunjungan": "jarang",
      "keluhan": "ya", "promo": "tinggi"}
x2 = {"tingkat": "perunggu", "kunjungan": "jarang",
      "keluhan": "ya", "promo": "rendah"}
print("x1         ->", naive_bayes(mini, x1))
print("x2         ->", naive_bayes(mini, x2))
print("x2 Laplace ->", naive_bayes(mini, x2, alfa=1))

## Listing 8.2: Gaussian naive Bayes untuk churn

`kode/bab08/nb_gauss.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
fitur = ["recency_hari", "frekuensi_90h", "nilai_90h", "keluhan_90h",
         "ragam_menu", "porsi_promo"]
X, y = df[fitur], df["churn"]
nb = GaussianNB().fit(X, y)
tab = pd.DataFrame(nb.theta_, columns=fitur, index=["setia", "churn"])
print("rata-rata per kelas:")
print(tab.T.round(2).to_string())
print("simpangan baku recency:", np.sqrt(nb.var_[:, 0]).round(2))

for nama, Xu in [("asli", X),
                 ("log1p", np.log1p(X))]:
    s = cross_val_score(GaussianNB(), Xu, y, cv=5, scoring="roc_auc")
    print(f"AUC CV, {nama:6s}: {s.mean():.3f}")

r = df["recency_hari"].to_numpy()
grid = np.arange(0, 90, 5.0) + 2.5
keluar = {"x": grid}
for c, nama in [(0, "setia"), (1, "churn")]:
    m, s = nb.theta_[c, 0], np.sqrt(nb.var_[c, 0])
    keluar["g_" + nama] = np.exp(-(grid - m) ** 2 / (2 * s * s)) / (
        s * np.sqrt(2 * np.pi))
    h, _ = np.histogram(r[y == c], bins=np.arange(0, 91, 5), density=True)
    keluar["h_" + nama] = h
pd.DataFrame(keluar).round(5).to_csv("gambar/bab08/recency.dat", sep=" ",
                                     index=False)

## Listing 8.3: Frekuensi nol pada tingkat keanggotaan

`kode/bab08/nb_kategorik.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv").sample(400, random_state=0)
print(pd.crosstab(df["tingkat"], df["churn"]))
X = df["tingkat"].map({"perunggu": 0, "perak": 1, "emas": 2}).to_frame()
for alfa in [1e-10, 1.0]:
    nb = CategoricalNB(alpha=alfa).fit(X, df["churn"])
    p = np.exp(nb.feature_log_prob_[0][1])
    print(f"alfa={alfa:g}: P(tingkat|churn) perunggu {p[0]:.4f}, "
          f"perak {p[1]:.4f}, emas {p[2]:.4f}")
    x = pd.DataFrame({"tingkat": [2]})
    print(f"         P(churn | emas) = {nb.predict_proba(x)[0, 1]:.4f}")

## Listing 8.4: Kurva belajar naive Bayes dan regresi logistik

`kode/bab08/nb_vs_logistik.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from kartu import kartu, y

Xl, Xu, yl, yu = train_test_split(kartu, y, test_size=0.4, stratify=y,
                                  random_state=1)
k = [int(kartu[c].max()) + 1 for c in kartu.columns]
hasil = []
for n in [20, 40, 80, 160, 320, 640, 1280, 2560, 4800]:
    a, b = [], []
    for r in range(30):
        i = Xl.sample(n, random_state=r).index
        if yl[i].nunique() < 2:
            continue
        nb = CategoricalNB(alpha=1, min_categories=k).fit(Xl.loc[i], yl[i])
        lg = make_pipeline(OneHotEncoder(handle_unknown="ignore"),
                           LogisticRegression(max_iter=2000)).fit(
            Xl.loc[i], yl[i])
        a.append(roc_auc_score(yu, nb.predict_proba(Xu)[:, 1]))
        b.append(roc_auc_score(yu, lg.predict_proba(Xu)[:, 1]))
    hasil.append((n, np.mean(a), np.mean(b)))
tab = pd.DataFrame(hasil, columns=["n", "nb", "log"])
print(tab.round(3).to_string(index=False))
tab.to_csv("gambar/bab08/kurva_belajar.dat", sep=" ", index=False)

## Listing 8.5: Penyiapan atribut kartu skor

`kode/bab08/kartu.py`

In [ ]:
"""Modul bantu Bab 8: atribut anggota yang didiskretkan untuk kartu skor."""
import numpy as np
import pandas as pd

DATA = "kode/data/"
df = pd.read_csv(DATA + "pelanggan.csv")
df["tren"] = df["frekuensi_90h"] / (df["frekuensi_90h_lalu"] + 1)
y = df["churn"]

BATAS = {"recency_hari": [7, 14, 30, 45],
         "tren": [0.5, 0.8, 1.1],
         "keluhan_90h": [0.5, 1.5],
         "ragam_menu": [2.5, 4.5, 6.5],
         "porsi_promo": [0.2, 0.4],
         "pakai_ulasan": [0.5]}
kartu = pd.DataFrame({k: np.digitize(df[k], b) for k, b in BATAS.items()})
kartu["tingkat"] = df["tingkat"].map({"perunggu": 0, "perak": 1,
                                      "emas": 2})

## Listing 8.6: Studi kasus kartu skor churn

`kode/bab08/studi_kasus.py`

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from kartu import df, kartu, y

cv = StratifiedKFold(10, shuffle=True, random_state=0)
semua = ["umur", "frekuensi_90h", "frekuensi_90h_lalu", "recency_hari",
         "nilai_90h", "rata_nilai_struk", "porsi_promo", "porsi_aplikasi",
         "porsi_kopi_hitam", "porsi_pagi", "porsi_akhir_pekan",
         "ragam_menu", "keluhan_90h", "jarak_km", "pakai_ulasan"]
logistik = make_pipeline(ColumnTransformer([
    ("n", StandardScaler(), semua),
    ("k", OneHotEncoder(), ["kota", "tingkat", "jenis_kelamin"])]),
    LogisticRegression(max_iter=2000))
calon = {"Gaussian NB, 15 atribut": (GaussianNB(), df[semua]),
         "NB kartu, 7 atribut": (CategoricalNB(alpha=1), kartu),
         "logistik, 18 atribut": (logistik, df)}
for nama, (m, X) in calon.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    print(f"{nama:24s} AUC {s.mean():.3f} +/- {s.std():.3f}")

nb = CategoricalNB(alpha=1).fit(kartu, y)
print("\nKartu skor (poin = 10 x log10 P(x|churn)/P(x|setia)):")
for j, atr in enumerate(kartu.columns):
    lp = nb.feature_log_prob_[j]
    poin = np.round(10 * (lp[1] - lp[0]) / np.log(10)).astype(int)
    print(f"  {atr:13s}", " ".join(f"{p:+d}" for p in poin))
awal = 10 * np.log10(y.mean() / (1 - y.mean()))
print(f"poin awal (peluang prior): {awal:.0f}")

poin_total = np.full(len(kartu), awal)
for j, atr in enumerate(kartu.columns):
    lp = nb.feature_log_prob_[j]
    poin = np.round(10 * (lp[1] - lp[0]) / np.log(10))
    poin_total += poin[kartu[atr].to_numpy()]
kelompok = np.digitize(poin_total, [-30, -20, -10, 0, 10])
nama = ["< -30", "-30..-21", "-20..-11", "-10..-1", "0..9", ">= 10"]
print("\npoin total  anggota  churn nyata  peluang NB")
for g in range(6):
    m = kelompok == g
    p_nb = 1 / (1 + 10 ** (-poin_total[m] / 10))
    print(f"{nama[g]:>9s}   {m.sum():6d}      {y[m].mean():.3f}"
          f"       {p_nb.mean():.3f}")